# Logit Lens and Tuned Lens

Lens methods show how a model's prediction develops across transformer blocks. `AllLayersSplitter` captures every residual-stream state in one trace, and the lens projects all of them together.

In [ ]:
from interpreto import AllLayersSplitter, LogitLens, TunedLens, plot_lens

## Logit Lens

The splitter can load a causal language model directly. Its `activation_names` list describes the order of the returned lens results.

In [ ]:
splitter = AllLayersSplitter("hf-internal-testing/tiny-random-gpt2")
text = "Interpreto is useful."

logit_lens = LogitLens(splitter, top_k=3)
logit_results = logit_lens(text)
list(logit_results)

In [ ]:
plot_lens(logit_results, text, tokenizer=splitter.tokenizer)

## Tuned Lens

A Tuned Lens learns one residual affine translator for each non-final state. The translators are trained jointly against the model's final prediction distribution.

In [ ]:
training_texts = [
    "Interpreto is useful.",
    "Lens methods expose intermediate predictions.",
]

tuned_lens = TunedLens(splitter, top_k=3)
losses = tuned_lens.fit(training_texts, epochs=1)
tuned_results = tuned_lens(text)
losses

`TunedLens` is a regular PyTorch module. Save and restore its translators with `state_dict()` and `load_state_dict()`.